# Dataset exploration

> **Training has moved to `src/train.py`** (`python -m src.train` from the project root).
> The cells below explore the raw PhishTank and Tranco data. The legitimate set they build
> (`https://<tranco-domain>` homepages) is **no longer used for training**: every legit URL had
> an empty path, so the old model learned "has a path => phishing". See `src/data.py` for the
> current sources and `models/model_meta.json` for the latest metrics.


In [1]:
import pandas as pd

phish_df = pd.read_csv("../data/raw/phishtank.csv")

print("Shape:", phish_df.shape)
phish_df.head()

Shape: (73267, 8)


,phish_id,url,phish_detail_url,submission_time,verified,verification_time,online,target
0,9510202,https://processesene.dressel-garten.de/f069d79...,http://www.phishtank.com/phish_detail.php?phis...,2026-08-24T13:58:50+00:00,yes,2026-08-24T14:03:07+00:00,yes,Other
1,9510198,https://bhdactivarusuario2026.weebly.com/,http://www.phishtank.com/phish_detail.php?phis...,2026-08-24T13:52:12+00:00,yes,2026-08-24T14:03:07+00:00,yes,Other
2,9510197,https://validaracceso-bhdactivarusuario.iceiy.com,http://www.phishtank.com/phish_detail.php?phis...,2026-08-24T13:51:51+00:00,yes,2026-08-24T14:03:07+00:00,yes,Other
3,9510195,https://encuestaecuador.lat/3gKhsRHNWn?adset_n...,http://www.phishtank.com/phish_detail.php?phis...,2026-08-24T13:51:35+00:00,yes,2026-08-24T14:03:07+00:00,yes,Other
4,9510187,https://cliente-a-limpfeirao.shop/,http://www.phishtank.com/phish_detail.php?phis...,2026-08-24T13:42:29+00:00,yes,2026-08-24T13:46:12+00:00,yes,Other


In [2]:
phish_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73267 entries, 0 to 73266
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   phish_id           73267 non-null  int64
 1   url                73267 non-null  str  
 2   phish_detail_url   73267 non-null  str  
 3   submission_time    73267 non-null  str  
 4   verified           73267 non-null  str  
 5   verification_time  73267 non-null  str  
 6   online             73267 non-null  str  
 7   target             73267 non-null  str  
dtypes: int64(1), str(7)
memory usage: 4.5 MB


In [3]:
phish_df.columns

Index(['phish_id', 'url', 'phish_detail_url', 'submission_time', 'verified',
       'verification_time', 'online', 'target'],
      dtype='str')

In [4]:
phish_df.isnull().sum()

phish_id             0
url                  0
phish_detail_url     0
submission_time      0
verified             0
verification_time    0
online               0
target               0
dtype: int64

In [5]:
phish_df["online"].value_counts()

online
yes    73267
Name: count, dtype: int64

In [6]:
phish_df["verified"].value_counts()

verified
yes    73267
Name: count, dtype: int64

In [7]:
print("Duplicate URLs:", phish_df["url"].duplicated().sum())

Duplicate URLs: 3


In [8]:
print("Unique URLs:", phish_df["url"].nunique())

Unique URLs: 73264


In [9]:
phish_df["target"].value_counts().head(20)

target
Other                                  65277
Allegro                                 3566
Internal Revenue Service                 880
Amazon.com                               600
Facebook                                 411
Microsoft                                374
Bradesco                                 236
Netflix                                  220
Optus                                    151
SociÃ©tÃ© GÃ©nÃ©rale                     146
Comcast                                  124
PayPal                                   102
Sumitomo Mitsui Banking Corporation       74
Adobe                                     71
Google                                    69
AT&amp;T                                  66
American Express                          66
Apple                                     61
DocuSign                                  60
DHL                                       49
Name: count, dtype: int64

In [10]:
phish_df = phish_df[
    (phish_df["verified"] == "yes") &
    (phish_df["online"] == "yes")
]

phish_df = phish_df.drop_duplicates(subset="url")

In [11]:
print("Empty URLs:", (phish_df["url"].str.strip() == "").sum())

Empty URLs: 0


In [12]:
print(phish_df["url"].head(10).to_string(index=False))

https://processesene.dressel-garten.de/f069d796...
         https://bhdactivarusuario2026.weebly.com/
 https://validaracceso-bhdactivarusuario.iceiy.com
https://encuestaecuador.lat/3gKhsRHNWn?adset_na...
                https://cliente-a-limpfeirao.shop/
              http://www.cliente-a-limpfeirao.shop
https://security-server-page--hollywoodcoache.r...
https://secure-site-manager--type4life.replit.app/
   https://secure-static-site--dgtoney.replit.app/
https://lndosnr743.shop/o/rsswb/732182999258316...


In [13]:
print(phish_df["url"].str[:20].value_counts().head(20))

url
https://docs.google.    5726
http://allegrolokaln    1219
https://sites.google    1140
https://new.express.    1069
https://allegrolokal     735
https://www.dropbox.     622
https://l.wl.co/l?u=     483
https://l.ead.me/bgb     471
https://l.ead.me/bga     375
https://tinyurl.com/     344
https://l.ead.me/bfJ     256
http://allegro-lokal     255
https://klelnanzeige     243
https://kleinanzeige     239
https://ipfs.best-pr     232
https://eu.jotform.c     216
https://l.ead.me/bfI     205
https://l.ead.me/bgc     184
https://qr-codes.io/     173
https://app.jotform.     165
Name: count, dtype: int64


In [14]:
phish_clean = phish_df[
    (phish_df["verified"] == "yes") &
    (phish_df["online"] == "yes")
].copy()

phish_clean = phish_clean.drop_duplicates(subset="url")

phish_clean = phish_clean[["url"]]

phish_clean["label"] = 1

In [15]:
tranco_df = pd.read_csv(
    "../data/raw/tranco.csv",
    header=None,
    names=["rank", "domain"]
)

print("Shape:", tranco_df.shape)
tranco_df.head()

Shape: (1000000, 2)


,rank,domain
0,1,google.com
1,2,cloudflare.com
2,3,facebook.com
3,4,gstatic.com
4,5,googleapis.com


In [16]:
tranco_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 2 columns):
 #   Column  Non-Null Count    Dtype
---  ------  --------------    -----
 0   rank    1000000 non-null  int64
 1   domain  1000000 non-null  str  
dtypes: int64(1), str(1)
memory usage: 15.3 MB


In [17]:
print("Duplicate domains:", tranco_df["domain"].duplicated().sum())
print("Unique domains:", tranco_df["domain"].nunique())

Duplicate domains: 0
Unique domains: 1000000


In [18]:
from urllib.parse import urlparse

# Extract hostname from phishing URLs
phish_domains = (
    phish_df["url"]
    .apply(lambda x: urlparse(x).netloc.lower().replace("www.", ""))
)

# Normalize Tranco domains
tranco_domains = (
    tranco_df["domain"]
    .str.lower()
    .str.strip()
    .str.replace(r"\.$", "", regex=True)
)

overlap = set(phish_domains) & set(tranco_domains)

print("Phishing domains:", len(set(phish_domains)))
print("Tranco domains:", len(set(tranco_domains)))
print("Overlapping domains:", len(overlap))

Phishing domains: 38429
Tranco domains: 1000000
Overlapping domains: 317


In [19]:
tranco_clean = tranco_df[
    ~tranco_domains.isin(overlap)
].copy()

print("Original Tranco domains:", len(tranco_df))
print("Removed overlapping domains:", len(overlap))
print("Remaining domains:", len(tranco_clean))

Original Tranco domains: 1000000
Removed overlapping domains: 317
Remaining domains: 999683


In [20]:
tranco_clean.head()

,rank,domain
1,2,cloudflare.com
2,3,facebook.com
3,4,gstatic.com
4,5,googleapis.com
5,6,microsoft.com


In [21]:
legit_candidates = tranco_clean.head(60000).copy()

legit_candidates["url"] = "https://" + legit_candidates["domain"]

print("Candidate URLs:", len(legit_candidates))
legit_candidates.head()

Candidate URLs: 60000


,rank,domain,url
1,2,cloudflare.com,https://cloudflare.com
2,3,facebook.com,https://facebook.com
3,4,gstatic.com,https://gstatic.com
4,5,googleapis.com,https://googleapis.com
5,6,microsoft.com,https://microsoft.com


In [22]:
test_candidates = legit_candidates.head(500).copy()

test_candidates["url"].head()

1    https://cloudflare.com
2      https://facebook.com
3       https://gstatic.com
4    https://googleapis.com
5     https://microsoft.com
Name: url, dtype: str

In [23]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def check_url(url):
    try:
        response = requests.get(
            url,
            timeout=5,
            allow_redirects=True,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        return {
            "url": url,
            "active": response.status_code < 500,
            "status_code": response.status_code,
            "error": None
        }

    except requests.RequestException as e:
        return {
            "url": url,
            "active": False,
            "status_code": None,
            "error": type(e).__name__
        }

In [24]:
results = []

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [
        executor.submit(check_url, url)
        for url in test_candidates["url"]
    ]

    for i, future in enumerate(as_completed(futures), 1):
        results.append(future.result())

        if i % 50 == 0:
            print(f"Checked {i}/500")

legit_status = pd.DataFrame(results)

print("Total checked:", len(legit_status))
print("Active:", legit_status["active"].sum())
print("Inactive:", (~legit_status["active"]).sum())

Checked 50/500
Checked 100/500
Checked 150/500
Checked 200/500
Checked 250/500
Checked 300/500
Checked 350/500
Checked 400/500
Checked 450/500
Checked 500/500
Total checked: 500
Active: 314
Inactive: 186


In [25]:
print(
    legit_status["status_code"]
    .value_counts(dropna=False)
)

status_code
200.0    229
NaN      185
403.0     58
404.0     13
202.0      7
401.0      3
400.0      1
418.0      1
429.0      1
498.0      1
503.0      1
Name: count, dtype: int64


In [26]:
legit_status["reachable"] = legit_status["status_code"].notna()

print(legit_status["reachable"].value_counts())

reachable
True     315
False    185
Name: count, dtype: int64


In [27]:
legit_reachable = legit_status[
    legit_status["reachable"]
].copy()

print("Reachable legitimate domains:", len(legit_reachable))

Reachable legitimate domains: 315


In [28]:
print(
    legit_reachable["status_code"]
    .value_counts()
)

status_code
200.0    229
403.0     58
404.0     13
202.0      7
401.0      3
400.0      1
418.0      1
429.0      1
498.0      1
503.0      1
Name: count, dtype: int64


In [29]:
legit_reachable["label"] = 0

In [30]:
legit_reachable[["url", "label"]].to_csv(
    "../data/processed/legitimate_validated.csv",
    index=False
)

In [31]:
print(legit_reachable[["url", "label"]].head())
print(legit_reachable["label"].value_counts())

                      url  label
0  https://googleapis.com      0
3       https://apple.com      0
4     https://gstatic.com      0
5   https://microsoft.com      0
6  https://cloudflare.com      0
label
0    315
Name: count, dtype: int64


In [32]:
phish_df["label"] = 1

print(phish_df[["url", "label"]].head())
print(phish_df["label"].value_counts())

                                                 url  label
0  https://processesene.dressel-garten.de/f069d79...      1
1          https://bhdactivarusuario2026.weebly.com/      1
2  https://validaracceso-bhdactivarusuario.iceiy.com      1
3  https://encuestaecuador.lat/3gKhsRHNWn?adset_n...      1
4                 https://cliente-a-limpfeirao.shop/      1
label
1    73264
Name: count, dtype: int64


In [33]:
print("Phishing URLs:", len(phish_df))
print("Unique phishing URLs:", phish_df["url"].nunique())
print("Duplicate phishing URLs:", phish_df["url"].duplicated().sum())

Phishing URLs: 73264
Unique phishing URLs: 73264
Duplicate phishing URLs: 0


In [34]:
phish_df = phish_df.drop_duplicates(subset="url").copy()

print("Phishing URLs:", len(phish_df))
print("Unique phishing URLs:", phish_df["url"].nunique())

Phishing URLs: 73264
Unique phishing URLs: 73264


In [35]:
overlap = set(phish_df["url"]) & set(legit_reachable["url"])

print("Exact URL overlap:", len(overlap))

Exact URL overlap: 0


In [36]:
legit_final = legit_reachable[["url", "label"]].copy()
phish_final = phish_df[["url", "label"]].copy()

dataset = pd.concat(
    [legit_final, phish_final],
    ignore_index=True
)

print("Final dataset shape:", dataset.shape)
print(dataset["label"].value_counts())

Final dataset shape: (73579, 2)
label
1    73264
0      315
Name: count, dtype: int64


In [37]:
legit_sample = tranco_df.sample(
    n=30000,
    random_state=42
).copy()

legit_sample["url"] = "https://" + legit_sample["domain"]

print("Candidate URLs:", len(legit_sample))
print(legit_sample.head())

Candidate URLs: 30000
          rank              domain                         url
987231  987232  inspections.gov.ua  https://inspections.gov.ua
79954    79955         babechat.ai         https://babechat.ai
567130  567131  alphaluckyworld.nl  https://alphaluckyworld.nl
500891  500892            dxzq.net            https://dxzq.net
55399    55400    memorialcare.org    https://memorialcare.org


In [38]:
phish_domains = set(
    phish_df["url"]
    .str.extract(r"https?://([^/]+)")[0]
    .str.lower()
)

legit_sample = legit_sample[
    ~legit_sample["domain"].str.lower().isin(phish_domains)
].copy()

print("After overlap removal:", len(legit_sample))

After overlap removal: 29994


In [39]:
test_sample = legit_sample.head(1000).copy()

print("Testing:", len(test_sample))

Testing: 1000


In [40]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

def check_url(url):
    try:
        response = requests.get(
            url,
            timeout=5,
            headers={
                "User-Agent": "Mozilla/5.0"
            },
            allow_redirects=True
        )

        return {
            "url": url,
            "status_code": response.status_code,
            "final_url": response.url
        }

    except requests.RequestException:
        return {
            "url": url,
            "status_code": None,
            "final_url": None
        }


results = []

with ThreadPoolExecutor(max_workers=20) as executor:

    futures = [
        executor.submit(check_url, url)
        for url in test_sample["url"]
    ]

    for i, future in enumerate(as_completed(futures), 1):

        results.append(future.result())

        if i % 50 == 0:
            print(f"Checked {i}/{len(futures)}")


test_status = pd.DataFrame(results)

print("\nTotal checked:", len(test_status))
print(
    "Responded:",
    test_status["status_code"].notna().sum()
)
print(
    "No response:",
    test_status["status_code"].isna().sum()
)

Checked 50/1000
Checked 100/1000
Checked 150/1000
Checked 200/1000
Checked 250/1000
Checked 300/1000
Checked 350/1000
Checked 400/1000
Checked 450/1000
Checked 500/1000
Checked 550/1000
Checked 600/1000
Checked 650/1000
Checked 700/1000
Checked 750/1000
Checked 800/1000
Checked 850/1000
Checked 900/1000
Checked 950/1000
Checked 1000/1000

Total checked: 1000
Responded: 758
No response: 242


In [41]:
print(
    test_status["status_code"]
    .value_counts(dropna=False)
)

status_code
200.0    589
NaN      242
403.0    135
404.0     11
406.0      5
500.0      4
503.0      3
401.0      2
202.0      2
525.0      1
468.0      1
521.0      1
502.0      1
520.0      1
429.0      1
523.0      1
Name: count, dtype: int64


In [42]:
print(
    test_status[test_status["status_code"].notna()]
    ["status_code"]
    .value_counts()
)

status_code
200.0    589
403.0    135
404.0     11
406.0      5
500.0      4
503.0      3
401.0      2
202.0      2
525.0      1
468.0      1
521.0      1
502.0      1
520.0      1
429.0      1
523.0      1
Name: count, dtype: int64


In [43]:
legit_validated = test_status[
    test_status["status_code"].notna()
].copy()

legit_validated["label"] = 0

In [44]:
legit_reachable = test_status[
    test_status["status_code"].notna()
].copy()

print("Reachable legitimate URLs:", len(legit_reachable))
print(legit_reachable[["url", "status_code"]].head())

Reachable legitimate URLs: 758
                         url  status_code
2         https://mairie.biz        403.0
3       https://novolink.net        403.0
4     https://motorfloor.com        200.0
6  https://rantingtogel.live        403.0
7      https://247horses.com        403.0


In [46]:
print("legit_reachable:", len(legit_reachable))
print("phishing:", len(phish_df))

legit_reachable: 758
phishing: 73264


In [47]:
# Balance the dataset

legit_final = legit_reachable.copy()

phish_final = phish_df.sample(
    n=len(legit_final),
    random_state=42
).copy()

dataset = pd.concat(
    [legit_final, phish_final],
    ignore_index=True
)

# Shuffle the combined dataset
dataset = dataset.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Dataset shape:", dataset.shape)
print("\nClass distribution:")
print(dataset["label"].value_counts())

Dataset shape: (1516, 11)

Class distribution:
label
1.0    758
Name: count, dtype: int64


In [62]:
# Balance the dataset

# Legitimate URLs = class 0
legit_final = legit_reachable.copy()
legit_final["label"] = 0

# Phishing URLs = class 1
phish_final = phish_df.sample(
    n=len(legit_final),
    random_state=42
).copy()
phish_final["label"] = 1

# Combine both classes
dataset = pd.concat(
    [legit_final, phish_final],
    ignore_index=True
)

# Shuffle
dataset = dataset.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Dataset shape:", dataset.shape)
print("\nClass distribution:")
print(dataset["label"].value_counts())

Dataset shape: (1516, 11)

Class distribution:
label
0    758
1    758
Name: count, dtype: int64


In [63]:
dataset.head(10)

,url,status_code,final_url,label,phish_id,phish_detail_url,submission_time,verified,verification_time,online,target
0,https://suteba.org.ar,200.0,https://www.suteba.org.ar/,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://recoverfit.com,200.0,https://recoverfit.com/,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://bit.ly/2210strp5,NaN,NaN,1,8823131.0,http://www.phishtank.com/phish_detail.php?phis...,2024-10-29T12:09:32+00:00,yes,2024-10-29T12:22:26+00:00,yes,Other
3,https://q-r.to/bgbaMo,NaN,NaN,1,9339147.0,http://www.phishtank.com/phish_detail.php?phis...,2026-02-10T00:21:19+00:00,yes,2026-02-10T00:41:59+00:00,yes,Other
4,https://copyrightaid.co.uk,200.0,https://copyrightaid.co.uk/,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,https://dgpugzqswtczge.web.app/,NaN,NaN,1,9250124.0,http://www.phishtank.com/phish_detail.php?phis...,2025-10-25T10:21:42+00:00,yes,2025-10-25T10:32:39+00:00,yes,Other
6,https://889993611.xyz,200.0,https://889993611.xyz/,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,https://mybroadband.co.za,403.0,https://mybroadband.co.za/,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,https://livedesktop-enus.wixstudio.com/desktop,NaN,NaN,1,9446159.0,http://www.phishtank.com/phish_detail.php?phis...,2026-06-07T10:19:20+00:00,yes,2026-06-07T10:22:38+00:00,yes,Other
9,https://www.pcn-noticey.top/com,NaN,NaN,1,9430818.0,http://www.phishtank.com/phish_detail.php?phis...,2026-05-21T22:53:50+00:00,yes,2026-05-21T23:52:23+00:00,yes,Other


## Current model

Features come from `src/features.py`, the same code the API uses, so there is no separate
copy to keep in sync. Artifacts are produced by `src/train.py`.


In [ ]:
import sys
import json
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.model import model, explainer, feature_columns, meta, predict_url

print("Model:", meta["model"], "| threshold:", meta["threshold"], "| trained:", meta["trained_at"])
print(json.dumps(meta["test"], indent=2))
print(json.dumps(meta["sanity"], indent=2))


In [ ]:
import numpy as np
import pandas as pd
import shap

from src.data import build_dataset
from src.features import extract_url_features

sample = build_dataset().sample(1000, random_state=0)
X_sample = pd.DataFrame([extract_url_features(u) for u in sample["url"]])[feature_columns]

shap_values = np.asarray(explainer.shap_values(X_sample))
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

shap.summary_plot(shap_values, X_sample, feature_names=feature_columns)


In [ ]:
def explain(url):
    result = predict_url(url)
    print(result["prediction"], f"{result['phishing_probability']:.1%}", url)

    contributions = pd.Series(result["contributions"])
    base = np.ravel(explainer.expected_value)[-1]
    shap.plots.waterfall(
        shap.Explanation(
            values=contributions.values,
            base_values=base,
            data=pd.Series(result["features"])[feature_columns].values,
            feature_names=feature_columns,
        ),
        max_display=12,
    )


explain("https://github.com/psf/requests")
explain("http://www.support-appleld.com.secureupdate.duilawyeryork.com/ap/de039a0b81784a8/?cmd=_update")
